In [ ]:
from pathlib import Path
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

data_path = Path("data/operation_.csv")
output_dir = Path("results")
output_dir.mkdir(parents=True, exist_ok=True)
data = pd.read_csv(data_path)


In [ ]:
outcomes = [
    "death_30d", "have_icu", "have_aki", "have_ali",
    "postop_lung_complications", "postop_stroke", "postop_cardiac_complications",
]
scores = ["asa", "sort_score", "CCI_score", "RCRI_score"]
train = data.loc[data["dataset"].eq(1)]
test = data.loc[data["dataset"].eq(2)]


In [ ]:
summary = []
for outcome in outcomes:
    for score in scores:
        train_rows = train[[outcome, score]].dropna()
        test_rows = test[["op_id", outcome, score]].dropna()
        if train_rows[outcome].nunique() < 2 or test_rows[outcome].nunique() < 2:
            continue
        model = LogisticRegression(max_iter=1000)
        model.fit(train_rows[[score]], train_rows[outcome].astype(int))
        probability = model.predict_proba(test_rows[[score]])[:, 1]
        predictions = pd.DataFrame({
            "op_id": test_rows["op_id"],
            "y_true": test_rows[outcome].astype(int),
            "y_pred_prob_0": probability,
        })
        predictions.to_csv(output_dir / f"preop_{score}_{outcome}.csv", index=False)
        summary.append({"outcome": outcome, "score": score, "auc": roc_auc_score(predictions["y_true"], probability)})

pd.DataFrame(summary).sort_values(["outcome", "score"])
